# Dependencies

In [5]:
!pip install pytest pytest-asyncio ipytest google-cloud-aiplatform -q

In [51]:
import google.genai as genai
from google.genai import types
from google.api_core.client_options import ClientOptions
import os
import ipytest
import pytest
from unittest.mock import MagicMock, patch
import vertexai
from vertexai.preview.evaluation import EvalTask
import pandas as pd

ipytest.autoconfig()

project_id = "qwiklabs-gcp-01-2fecb47c3bc2"
location_id = "us-central1"

genai_client = genai.Client(
    vertexai=True,
    project=project_id,
    location=location_id
)

vertexai.init(project=project_id, location=location_id)

# Classify a question

In [45]:
def classify_question(user_question):
    response = genai_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_question,
        config={
            "system_instruction": """
            You are a government question classifier.
            Classify the user's question into EXACTLY ONE of these categories:
            - Employment
            - General Information
            - Emergency Services
            - Tax Related
            Respond with ONLY the category name, nothing else.
            """
        }
    )
    return response.text.strip()

# Social media post generation

In [46]:
def generate_social_media_post(announcement, platform="general"):
    response = genai_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=announcement,
        config={
            "system_instruction": f"""
            You are a government social media manager.
            Generate a clear, professional social media post for a government announcement.
            Platform: {platform}
            Guidelines:
            - Twitter/X: max 280 characters, use relevant hashtags
            - Facebook: conversational, 1-3 short paragraphs, include hashtags
            - General: concise, professional, include hashtags
            Use an urgent but calm tone for emergencies.
            Use a friendly tone for holidays and closings.
            Always include relevant emojis and 2-3 hashtags.
            """
        }
    )
    return response.text.strip()

# Unit tests

In [47]:
%%ipytest -v

VALID_CATEGORIES = {"Employment", "General Information", "Emergency Services", "Tax Related"}

# ── classifier tests ──────────────────────────────────────────────────────────

@pytest.mark.parametrize("question,expected", [
    ("How do I file my taxes?", "Tax Related"),
    ("How do I apply for unemployment?", "Employment"),
    ("What do I do during a tornado warning?", "Emergency Services"),
    ("What are city hall's office hours?", "General Information"),
])
def test_classify_question_known_inputs(question, expected):
    result = classify_question(question)
    assert result == expected

def test_classify_question_returns_valid_category():
    result = classify_question("Can you help me with something?")
    assert result in VALID_CATEGORIES

def test_classify_question_strips_whitespace():
    result = classify_question("How do I pay property tax?")
    assert result == result.strip()

# ── social media tests ────────────────────────────────────────────────────────

@pytest.mark.parametrize("platform", ["Twitter/X", "Facebook", "general"])
def test_social_media_post_returns_string(platform):
    result = generate_social_media_post("City hall will be closed Monday.", platform)
    assert isinstance(result, str) and len(result) > 0

def test_social_media_twitter_length():
    result = generate_social_media_post(
        "All schools closed tomorrow due to snow.", "Twitter/X"
    )
    assert len(result) <= 280

def test_social_media_post_contains_hashtag():
    result = generate_social_media_post(
        "Boil water advisory in effect for downtown.", "general"
    )
    assert "#" in result

======================================= test session starts ========================================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: asyncio-1.4.0, anyio-4.13.0, langsmith-0.8.5, typeguard-4.5.2
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 11 items

t_e05a39ae8a5c462781451de7b0d20d2a.py ...........                                            [100%]

========================================= warnings summary =========================================
../usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1290
  /usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
============================= 11 passed, 1

# Prompt comparison

In [52]:
# Prompt A — terse, direct
PROMPT_A_TEMPLATE = """
Classify this government question into EXACTLY ONE of:
Employment, General Information, Emergency Services, Tax Related.
Respond with ONLY the category name.
Question: {question}
"""

# Prompt B — detailed, contextual
PROMPT_B_TEMPLATE = """
You are an expert government services router helping citizens find the right department.
Read the question carefully and determine which department handles it.
Departments available: Employment, General Information, Emergency Services, Tax Related.
Return only the department name with no explanation or punctuation.
Question: {question}
"""

test_questions = [
    "How do I file my taxes?",
    "What do I do during a tornado warning?",
    "How can I apply for unemployment benefits?",
    "What are the office hours for city hall?",
    "Is there a payment plan for overdue taxes?",
    "Who do I call for a gas leak?",
    "How do I register a new business?",
    "What is the process for appealing a tax decision?",
]

ground_truth = [
    "Tax Related",
    "Emergency Services",
    "Employment",
    "General Information",
    "Tax Related",
    "Emergency Services",
    "General Information",
    "Tax Related",
]

In [54]:
responses_a = []
responses_b = []

for q in test_questions:
    resp_a = genai_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=PROMPT_A_TEMPLATE.format(question=q)
    )
    responses_a.append(resp_a.text.strip())

    resp_b = genai_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=PROMPT_B_TEMPLATE.format(question=q)
    )
    responses_b.append(resp_b.text.strip())

print("Prompt A responses:", responses_a)
print("Prompt B responses:", responses_b)

Prompt A responses: ['Tax Related', 'Emergency Services', 'Employment', 'General Information', 'Tax Related', 'Emergency Services', 'General Information', 'Tax Related']
Prompt B responses: ['Tax Related', 'Emergency Services', 'Employment', 'General Information', 'Tax Related', 'Emergency Services', 'General Information', 'Tax Related']


In [55]:
eval_dataset_a = pd.DataFrame({
    "prompt":    [PROMPT_A_TEMPLATE.format(question=q) for q in test_questions],
    "response":  responses_a,
    "reference": ground_truth,
})

eval_dataset_b = pd.DataFrame({
    "prompt":    [PROMPT_B_TEMPLATE.format(question=q) for q in test_questions],
    "response":  responses_b,
    "reference": ground_truth,
})

print(eval_dataset_a)

                                              prompt             response  \
0  \nClassify this government question into EXACT...          Tax Related   
1  \nClassify this government question into EXACT...   Emergency Services   
2  \nClassify this government question into EXACT...           Employment   
3  \nClassify this government question into EXACT...  General Information   
4  \nClassify this government question into EXACT...          Tax Related   
5  \nClassify this government question into EXACT...   Emergency Services   
6  \nClassify this government question into EXACT...  General Information   
7  \nClassify this government question into EXACT...          Tax Related   

             reference  
0          Tax Related  
1   Emergency Services  
2           Employment  
3  General Information  
4          Tax Related  
5   Emergency Services  
6  General Information  
7          Tax Related  


In [57]:
metrics = ["rouge_l_sum", "bleu", "fluency", "coherence", "instruction_following"]

eval_task_a = EvalTask(
    dataset=eval_dataset_a,
    metrics=metrics,
    experiment="classifier-prompt-comparison",
)
result_a = eval_task_a.evaluate()

eval_task_b = EvalTask(
    dataset=eval_dataset_b,
    metrics=metrics,
    experiment="classifier-prompt-comparison",
)
result_b = eval_task_b.evaluate()

print("=== Prompt A Summary ===")
print(result_a.summary_metrics)

print("\n=== Prompt B Summary ===")
print(result_b.summary_metrics)

INFO:vertexai.preview.evaluation._evaluation:Computing metrics with a total of 40 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 40/40 [00:46<00:00,  1.17s/it]
INFO:vertexai.preview.evaluation._evaluation:All 40 metric requests are successfully computed.
INFO:vertexai.preview.evaluation._evaluation:Evaluation Took:46.91610134499933 seconds


INFO:vertexai.preview.evaluation._evaluation:Computing metrics with a total of 40 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 40/40 [00:38<00:00,  1.04it/s]
INFO:vertexai.preview.evaluation._evaluation:All 40 metric requests are successfully computed.
INFO:vertexai.preview.evaluation._evaluation:Evaluation Took:38.388016683002206 seconds


=== Prompt A Summary ===
{'row_count': 8, 'rouge_l_sum/mean': np.float64(1.0), 'rouge_l_sum/std': 0.0, 'bleu/mean': np.float64(0.0), 'bleu/std': 0.0, 'fluency/mean': np.float64(5.0), 'fluency/std': 0.0, 'coherence/mean': np.float64(5.0), 'coherence/std': 0.0, 'instruction_following/mean': np.float64(5.0), 'instruction_following/std': 0.0}

=== Prompt B Summary ===
{'row_count': 8, 'rouge_l_sum/mean': np.float64(1.0), 'rouge_l_sum/std': 0.0, 'bleu/mean': np.float64(0.0), 'bleu/std': 0.0, 'fluency/mean': np.float64(5.0), 'fluency/std': 0.0, 'coherence/mean': np.float64(5.0), 'coherence/std': 0.0, 'instruction_following/mean': np.float64(5.0), 'instruction_following/std': 0.0}


In [58]:
metric_keys = [
    "rouge_l_sum/mean",
    "bleu/mean",
    "fluency/mean",
    "coherence/mean",
    "instruction_following/mean",
]

comparison = pd.DataFrame({
    "Metric":   metric_keys,
    "Prompt A": [result_a.summary_metrics.get(m, "N/A") for m in metric_keys],
    "Prompt B": [result_b.summary_metrics.get(m, "N/A") for m in metric_keys],
})

print("\n=== Head-to-Head Comparison ===")
print(comparison.to_string(index=False))

# Pick winner by coherence
score_a = result_a.summary_metrics.get("coherence/mean", 0)
score_b = result_b.summary_metrics.get("coherence/mean", 0)
winner = "Prompt A (terse)" if score_a >= score_b else "Prompt B (detailed)"
print(f"Winner by coherence: {winner}")


=== Head-to-Head Comparison ===
                    Metric  Prompt A  Prompt B
          rouge_l_sum/mean       1.0       1.0
                 bleu/mean       0.0       0.0
              fluency/mean       5.0       5.0
            coherence/mean       5.0       5.0
instruction_following/mean       5.0       5.0
Winner by coherence: Prompt A (terse)
